<a href="https://colab.research.google.com/github/Vivekb638/movie-recommendation-pyspark/blob/main/notebooks/01_data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "movie-recommendation-pyspark"
)

directories = [
    "data/raw",
    "data/processed",
    "models",
    "results",
    "logs"
]

for directory in directories:
    os.makedirs(
        os.path.join(PROJECT_DIR, directory),
        exist_ok=True
    )

print("Cloud storage configured successfully!")
print(PROJECT_DIR)

Cloud storage configured successfully!
/content/drive/MyDrive/movie-recommendation-pyspark


In [6]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip install -q pyspark==3.5.5

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
import os

os.environ["JAVA_HOME"] = (
    "/usr/lib/jvm/java-17-openjdk-amd64"
)

os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + "/bin:" +
    os.environ["PATH"]
)

!java -version

openjdk version "17.0.21.0.3" 2026-10-24
OpenJDK Runtime Environment (build 17.0.21.0.3+1-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.21.0.3+1-1-24.04.1-Ubuntu, mixed mode, sharing)


In [7]:
import pyspark

print("PySpark version:", pyspark.__version__)

PySpark version: 3.5.5


In [8]:
import urllib.request
import zipfile
import os

DATASET_URL = (
    "https://files.grouplens.org/"
    "datasets/movielens/ml-1m.zip"
)

RAW_DIR = os.path.join(
    PROJECT_DIR,
    "data/raw"
)

ZIP_PATH = os.path.join(
    RAW_DIR,
    "ml-1m.zip"
)

EXTRACT_DIR = os.path.join(
    RAW_DIR,
    "ml-1m"
)

os.makedirs(RAW_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading MovieLens 1M...")

    urllib.request.urlretrieve(
        DATASET_URL,
        ZIP_PATH
    )

    print("Download completed!")

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(
        ZIP_PATH, "r"
    ) as zip_ref:

        zip_ref.extractall(RAW_DIR)

    print("Dataset extracted!")

print("Dataset location:", EXTRACT_DIR)

Download completed!
Dataset extracted!
Dataset location: /content/drive/MyDrive/movie-recommendation-pyspark/data/raw/ml-1m


In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MovieRecommendationSystem")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Apache Spark started successfully!")

print("Spark version:", spark.version)

Apache Spark started successfully!
Spark version: 3.5.5


In [10]:
from pyspark.sql.functions import (
    col,
    split
)

DATA_PATH = EXTRACT_DIR

# Load ratings
ratings_raw = spark.read.text(
    f"{DATA_PATH}/ratings.dat"
)

ratings = ratings_raw.select(
    split(col("value"), "::").getItem(0)
        .cast("int").alias("userId"),

    split(col("value"), "::").getItem(1)
        .cast("int").alias("movieId"),

    split(col("value"), "::").getItem(2)
        .cast("float").alias("rating"),

    split(col("value"), "::").getItem(3)
        .cast("long").alias("timestamp")
)

# Load movies
movies_raw = spark.read.text(
    f"{DATA_PATH}/movies.dat"
)

movies = movies_raw.select(
    split(col("value"), "::").getItem(0)
        .cast("int").alias("movieId"),

    split(col("value"), "::").getItem(1)
        .alias("title"),

    split(col("value"), "::").getItem(2)
        .alias("genres")
)

# Load users
users_raw = spark.read.text(
    f"{DATA_PATH}/users.dat"
)

users = users_raw.select(
    split(col("value"), "::").getItem(0)
        .cast("int").alias("userId"),

    split(col("value"), "::").getItem(1)
        .alias("gender"),

    split(col("value"), "::").getItem(2)
        .cast("int").alias("age"),

    split(col("value"), "::").getItem(3)
        .cast("int").alias("occupation"),

    split(col("value"), "::").getItem(4)
        .alias("zipCode")
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [11]:
print("Total ratings:", ratings.count())

print("Total movies:", movies.count())

print("Total users:", users.count())

print("\nRATINGS")
ratings.show(5, truncate=False)

print("\nMOVIES")
movies.show(5, truncate=False)

print("\nUSERS")
users.show(5, truncate=False)

Total ratings: 1000209
Total movies: 3883
Total users: 6040

RATINGS
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1193   |5.0   |978300760|
|1     |661    |3.0   |978302109|
|1     |914    |3.0   |978301968|
|1     |3408   |4.0   |978300275|
|1     |2355   |5.0   |978824291|
+------+-------+------+---------+
only showing top 5 rows


MOVIES
+-------+----------------------------------+----------------------------+
|movieId|title                             |genres                      |
+-------+----------------------------------+----------------------------+
|1      |Toy Story (1995)                  |Animation|Children's|Comedy |
|2      |Jumanji (1995)                    |Adventure|Children's|Fantasy|
|3      |Grumpier Old Men (1995)           |Comedy|Romance              |
|4      |Waiting to Exhale (1995)          |Comedy|Drama                |
|5      |Father of the Bride Part II (1995)|Comedy                      |


In [12]:
PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    "data/processed"
)

ratings.write.mode("overwrite").parquet(
    f"{PROCESSED_DIR}/ratings"
)

movies.write.mode("overwrite").parquet(
    f"{PROCESSED_DIR}/movies"
)

users.write.mode("overwrite").parquet(
    f"{PROCESSED_DIR}/users"
)

print("All datasets saved in Parquet format!")

All datasets saved in Parquet format!


In [13]:
print(ratings.count())
print(movies.count())
print(users.count())

1000209
3883
6040
